## 6. Generative Models

### 1. Implement a Variational Autoencoder with TensorFlow/Keras.

In [4]:
import tensorflow as tf

LATENT_DIM = 2

# Encoder
encoder = tf.keras.Sequential([
    tf.keras.layers.Input((28, 28, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(2 * LATENT_DIM),
])

# Decoder
decoder = tf.keras.Sequential([
    tf.keras.layers.Input((LATENT_DIM,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(28 * 28, activation="sigmoid"),
    tf.keras.layers.Reshape((28, 28, 1))
])

class VAE(tf.keras.Model):
    def call(self, x):
        mean, log_var = tf.split(encoder(x), 2, axis=1)

        # Reparameterization 
        z = mean + tf.exp(0.5 * log_var) * tf.random.normal(tf.shape(mean))
        reconstruction = decoder(z)

        bce = tf.keras.losses.binary_crossentropy(x, reconstruction)
        reconstruction_loss = tf.reduce_sum(bce, axis=(1, 2))
        kl_loss = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mean) - tf.exp(log_var), axis=1) 
    
        self.add_loss(tf.reduce_mean(reconstruction_loss + kl_loss))
        return reconstruction

(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32')[..., None]/255.0

vae=VAE()
vae.compile(optimizer="adam")
vae.fit(x_train, x_train, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 545.3043
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 545.3060
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 545.3053
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 545.3033
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 545.3015
